In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, GRU , Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

c:\Users\Amir sohail\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df = pd.read_csv("subjects-questions.csv.")
df.head()

,eng,Subject
0,An anti-forest measure is\nA. Afforestation\nB...,Biology
1,"Among the following organic acids, the acid pr...",Chemistry
2,If the area of two similar triangles are equal...,Maths
3,"In recent year, there has been a growing\nconc...",Biology
4,Which of the following statement\nregarding tr...,Physics


In [3]:
df.columns

Index(['eng', 'Subject'], dtype='object')

In [4]:
df.shape

(122519, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122519 entries, 0 to 122518
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   eng      122519 non-null  object
 1   Subject  122519 non-null  object
dtypes: object(2)
memory usage: 1.9+ MB


In [6]:
df.isnull().sum()

eng        0
Subject    0
dtype: int64

In [7]:
df["Subject"].unique()

array(['Biology', 'Chemistry', 'Maths', 'Physics'], dtype=object)

In [8]:
df.duplicated().sum()

np.int64(811)

In [9]:
df["eng"].duplicated().sum()

np.int64(840)

In [10]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [11]:
df.shape

(121708, 2)

In [12]:
df.head()

,eng,Subject
0,An anti-forest measure is\nA. Afforestation\nB...,Biology
1,"Among the following organic acids, the acid pr...",Chemistry
2,If the area of two similar triangles are equal...,Maths
3,"In recent year, there has been a growing\nconc...",Biology
4,Which of the following statement\nregarding tr...,Physics


In [13]:
def tolower(txt):
    return txt.lower()

def remove_punctuation(txt):
    import string
    return txt.translate(str.maketrans("", "", string.punctuation))

def remove_emojis(txt):
    new_txt = ""
    for i in txt:
        if i.isascii():
            new_txt+=i
    return new_txt

In [14]:
label_encoder = LabelEncoder()

df["Subject"] = label_encoder.fit_transform(df["Subject"])
df.head()

,eng,Subject
0,An anti-forest measure is\nA. Afforestation\nB...,0
1,"Among the following organic acids, the acid pr...",1
2,If the area of two similar triangles are equal...,2
3,"In recent year, there has been a growing\nconc...",0
4,Which of the following statement\nregarding tr...,3


In [15]:
df["eng"] = df["eng"].apply(tolower)
df["eng"] = df["eng"].apply(remove_punctuation)
df["eng"] = df["eng"].apply(remove_emojis)

df.head()

,eng,Subject
0,an antiforest measure is\na afforestation\nb s...,0
1,among the following organic acids the acid pre...,1
2,if the area of two similar triangles are equal...,2
3,in recent year there has been a growing\nconce...,0
4,which of the following statement\nregarding tr...,3


In [16]:
X = df.drop("Subject",axis=1)
y = df[["Subject"]]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
Max_len = 100
Max_words = 10000
Embedding_dim = 100

In [18]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["eng"])
train_seq = tokenizer.texts_to_sequences(X_train["eng"])
train_X_pad = pad_sequences(train_seq, maxlen=Max_len, padding="post", truncating="post")
test_seq = tokenizer.texts_to_sequences(X_test["eng"])
test_X_pad = pad_sequences(test_seq, maxlen=Max_len, padding="post",truncating="post")

In [19]:
df["Subject"].unique()

array([0, 1, 2, 3])

In [20]:
model = Sequential([
    Embedding(input_dim=Max_words, output_dim=Embedding_dim, input_length=Max_len),
    Bidirectional(GRU(64,return_sequences=True)),
    Dropout(0.3),
    Bidirectional(GRU(64,return_sequences=False)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(4, activation="softmax"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.EarlyStopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
his = model.fit(train_X_pad, y_train, validation_data=(test_X_pad, y_test), epochs=2, batch_size=128)
model.summary()

Epoch 1/2
761/761 ━━━━━━━━━━━━━━━━━━━━ 384s 491ms/step - accuracy: 0.8621 - loss: 0.3651 - val_accuracy: 0.9186 - val_loss: 0.2254
Epoch 2/2
761/761 ━━━━━━━━━━━━━━━━━━━━ 429s 552ms/step - accuracy: 0.9309 - loss: 0.1929 - val_accuracy: 0.9198 - val_loss: 0.2220


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 100, 128)       │        63,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,428,894 (13.08 MB)

 Trainable params: 1,142,964 (4.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,285,930 (8.72 MB)

In [22]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
model.save("model.h5")
